# CTC Model Training Pipeline
This notebook implements the Connectionist Temporal Classification (CTC) pipeline. 
Unlike the sliding window approach, this trains the `CTCModel` sequentially on entire audio recordings using PyTorch's native `CTCLoss`.

In [ ]:
import sys

assert sys.version_info >= (3, 10)
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    !git clone https://github.com/stachuapa123/ASR_project.git
    %cd ASR_project
    # !git checkout <YOUR_BRANCH_NAME>  # Uncomment and set this to your branch if needed
    !pip install -q torchmetrics
    from google.colab import drive

    drive.mount("/content/drive")

    # Extract data securely if on Colab
    !mkdir -p "/content/asr_data"
    !unzip -q "/content/drive/MyDrive/asr_data.zip" -d "/content/asr_data"
    DATA_DIR = "/content/asr_data"
else:
    # Local path
    %load_ext autoreload
    %autoreload 2
    DATA_DIR = "../data"  # Update to your local subset or AutorskieDane

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.ctc.config import CTCConfig as C
from src.ctc.model import CTCModel
from src.ctc.dataset import CTCDataset, waveform_collate_fn
from src.ctc.features import CTCFeatureExtractor
from src.ctc.augmentation import SpecAugment
from src.ctc.training import train_ctc, EarlyStopping
from src.utils.device import get_device
from src.utils.speaker_split import load_splits

In [ ]:
# Hyperparameters
SPLITS_PATH = "../data/splits.json"  # shared speaker partition (run scripts.build_splits first)
MAX_FILES = None  # None = full corpus; set small for a quick smoke run
BATCH_SIZE = 32
N_EPOCHS = 100
LR = 1e-3
MAX_LR = 1e-3
WEIGHT_DECAY = 1e-4
PCT_START = 0.2
NUM_WORKERS = 4
EARLY_STOP_PATIENCE = 10
EARLY_STOP_MIN_DELTA = 1e-3
NUM_LAYERS = 3
HIDDEN_SIZE = 192
NORM_GROUPS = 8
CNN_DROPOUT = 0.2
RNN_DROPOUT = 0.3
HEAD_DROPOUT = 0.3

device = get_device()
print(f"Using device: {device}")

In [ ]:
# Speaker-disjoint split from data/splits.json. TRAIN augmented; VAL clean (honest PER).
splits = load_splits(SPLITS_PATH)
print(f"Speakers -> train {len(splits['train'])} | val {len(splits['val'])} | test {len(splits['test'])} (held out)")

# waveform mode: features (log-mel + augmentation) built on GPU by CTCFeatureExtractor
train_set = CTCDataset(
    data_root=DATA_DIR,
    speakers=set(splits["train"]),
    apply_augmentations=True,
    return_waveform=True,
    max_files=MAX_FILES,
)
val_set = CTCDataset(
    data_root=DATA_DIR,
    speakers=set(splits["val"]),
    apply_augmentations=False,  # clean val for an honest PER
    return_waveform=True,
    max_files=MAX_FILES,
)
print(f"Train items: {len(train_set)} | Val items: {len(val_set)}")

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=waveform_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=waveform_collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

In [ ]:
model = CTCModel(
    num_layers=NUM_LAYERS,
    hidden_size=HIDDEN_SIZE,
    cnn_dropout=CNN_DROPOUT,
    rnn_dropout=RNN_DROPOUT,
    head_dropout=HEAD_DROPOUT,
    norm_groups=NORM_GROUPS,
)
objective = torch.nn.CTCLoss(blank=C.BLANK_IDX, zero_infinity=True)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    steps_per_epoch=len(train_loader),
    epochs=N_EPOCHS,
    pct_start=PCT_START,
)
scaler = torch.amp.GradScaler(
    device=device.type,
    enabled=(device.type == "cuda"),
)
early_stopping = EarlyStopping(
    patience=EARLY_STOP_PATIENCE, min_delta=EARLY_STOP_MIN_DELTA
)
feature_extractor = CTCFeatureExtractor(spec_augment=SpecAugment())

In [ ]:
print("Learnable parameters: ", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
model = train_ctc(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    objective=objective,
    device=device,
    n_epochs=N_EPOCHS,
    feature_extractor=feature_extractor,
    scheduler=scheduler,
    scaler=scaler,
    early_stopping=early_stopping,
    save_best_to="../trained_models/ctc_speaker_disjoint.pt",
    use_amp=(device.type == "cuda"),
    step_scheduler_per_batch=True,
)